In [33]:
"""
SurveyMind Response Pattern Realism Study

HUMAN SIDE: REAL. Loads and cleans the Open Psychometrics Big Five dataset (data-final.csv) exactly per the paper's methodology.

SYNTHETIC SIDE: STILL DUMMY. SurveyMind's actual outputis not available. The synthetic data below is generated and NOT by
SurveyMind, purely to test the analysis pipeline end-to-end.

Once we have the data we can replace it with real SurveyMind.

"""

import pandas as pd
import numpy as np
from scipy import stats

np.random.seed(42)

RAW_PATH = "data-final.csv"
TRAITS = ["Openness", "Conscientiousness", "Extraversion", "Agreeableness", "Neuroticism"]

REVERSE_ITEMS = {
    "EXT2", "EXT4", "EXT6", "EXT8", "EXT10",
    "EST2", "EST4",
    "AGR1", "AGR3", "AGR5", "AGR7",
    "CSN2", "CSN4", "CSN6", "CSN8",
    "OPN2", "OPN4", "OPN6",
}

TRAIT_ITEMS = {
    "Extraversion": [f"EXT{i}" for i in range(1, 11)],
    "Neuroticism": [f"EST{i}" for i in range(1, 11)],
    "Agreeableness": [f"AGR{i}" for i in range(1, 11)],
    "Conscientiousness": [f"CSN{i}" for i in range(1, 11)],
    "Openness": [f"OPN{i}" for i in range(1, 11)],
}



In [24]:
# STEP 1: Load and clean the REAL human dataset

def load_and_clean_human_data():
    item_cols = sum(TRAIT_ITEMS.values(), [])

    raw_preview = pd.read_csv(RAW_PATH, sep="\t")
    print(raw_preview.shape)
    print(raw_preview.head())

    usecols = item_cols + ["IPC"]
    df = pd.read_csv(RAW_PATH, sep="\t", usecols=usecols)
    n_raw = len(df)

    df = df[df["IPC"] == 1]
    df = df[(df[item_cols] > 0).all(axis=1)]
    n_clean = len(df)

    for item in REVERSE_ITEMS:
        df[item] = 6 - df[item]

    trait_df = pd.DataFrame({
        trait: df[items].mean(axis=1)
        for trait, items in TRAIT_ITEMS.items()
    })[TRAITS]

    print(f"Raw rows: {n_raw:,}")
    print(f"After IPC==1 and complete-response filtering: {n_clean:,}")
    print(f"Final trait-score table shape: {trait_df.shape}")

    return trait_df


In [26]:
# STEP 2: Generate DUMMY synthetic data (placeholder for SurveyMind output)

def simulate_from_corr(corr_matrix, n, mean_vec, std_vec, seed):
    rng = np.random.default_rng(seed)
    latent = rng.multivariate_normal(mean=np.zeros(5), cov=corr_matrix, size=n)
    scaled = mean_vec + latent * std_vec
    return pd.DataFrame(np.clip(scaled, 1, 5), columns=TRAITS)



In [28]:
# STEP 3: Analysis functions

def tucker_congruence(r1, r2):
    iu = np.triu_indices_from(r1, k=1)
    v1, v2 = r1[iu], r2[iu]
    return np.sum(v1 * v2) / np.sqrt(np.sum(v1**2) * np.sum(v2**2))


def jennrich_test(r1, r2, n1, n2):
    p = r1.shape[0]
    n = (n1 * n2) / (n1 + n2)
    r_pooled = (n1 * r1 + n2 * r2) / (n1 + n2)
    iu = np.triu_indices(p, k=1)
    pairs = list(zip(iu[0], iu[1]))
    m = len(pairs)
    z = np.sqrt(n) * np.array([r1[i, j] - r2[i, j] for i, j in pairs])
    sigma = np.zeros((m, m))
    for a, (i, j) in enumerate(pairs):
        for b, (k, l) in enumerate(pairs):
            rik, rjl = r_pooled[i, k], r_pooled[j, l]
            ril, rjk = r_pooled[i, l], r_pooled[j, k]
            rij, rkl = r_pooled[i, j], r_pooled[k, l]
            sigma[a, b] = 0.5 * (
                (rik - rij * rjk) * (rjl - rij * ril)
                + (ril - rij * rjl) * (rjk - rij * rik)
            ) + 0.5 * (rik * rjl + ril * rjk) - rij * rkl
    sigma += np.eye(m) * 1e-8
    try:
        chi2_stat = z @ np.linalg.inv(sigma) @ z
    except np.linalg.LinAlgError:
        chi2_stat = np.nan
    p_value = 1 - stats.chi2.cdf(chi2_stat, m) if not np.isnan(chi2_stat) else np.nan
    return chi2_stat, m, p_value


def bootstrap_congruence_ci(synth_df, human_corr, n_boot=2000, seed=0):
    rng = np.random.default_rng(seed)
    n = len(synth_df)
    arr = synth_df[TRAITS].values
    vals = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        r_boot = np.corrcoef(arr[idx], rowvar=False)
        vals.append(tucker_congruence(human_corr, r_boot))
    vals = np.array(vals)
    return np.percentile(vals, 2.5), np.percentile(vals, 97.5)


def ks_marginal_tests(human_df, synth_df):
    return {t: stats.ks_2samp(human_df[t], synth_df[t]) for t in TRAITS}


def run_scenario(name, human_df, r_human, synth_df, n_human):
    print(f"\n{'='*70}\nSCENARIO: {name}\n{'='*70}")
    r_synth = synth_df[TRAITS].corr().values

    print(f"\nREAL human correlation matrix (n={n_human:,}):")
    print(pd.DataFrame(r_human, index=TRAITS, columns=TRAITS).round(3))

    print(f"\nDUMMY synthetic correlation matrix (n={len(synth_df)}):")
    print(pd.DataFrame(r_synth, index=TRAITS, columns=TRAITS).round(3))

    cc = tucker_congruence(r_human, r_synth)
    print(f"\nTucker's congruence coefficient: {cc:.4f}")
    print(f"Pre-registered threshold: >= 0.90 -> {'PASS' if cc >= 0.90 else 'FAIL'}")

    chi2_stat, df, p_val = jennrich_test(r_human, r_synth, n_human, len(synth_df))
    print(f"\nJennrich test: chi2 = {chi2_stat:.3f}, df = {df}, p = {p_val:.4f}")

    lo, hi = bootstrap_congruence_ci(synth_df, r_human)
    print(f"\nBootstrap 95% CI for congruence coefficient: [{lo:.4f}, {hi:.4f}]")

    print("\nMarginal distribution checks (KS test per trait):")
    for trait, (stat, p) in ks_marginal_tests(human_df, synth_df).items():
        flag = "differs" if p < 0.05 else "no significant difference"
        print(f"  {trait:18s} KS={stat:.3f}, p={p:.4f}  ({flag})")



In [30]:
# MAIN


if __name__ == "__main__":
    human_df = load_and_clean_human_data()
    r_human = human_df[TRAITS].corr().values
    n_human = len(human_df)

    mean_vec = human_df[TRAITS].mean().values
    std_vec = human_df[TRAITS].std().values
    N_SYNTH = 200

    synthetic_good_df = simulate_from_corr(r_human, N_SYNTH, mean_vec, std_vec, seed=2)

    distorted_corr = r_human.copy()
    distorted_corr[4, :] *= 0.2
    distorted_corr[:, 4] *= 0.2
    np.fill_diagonal(distorted_corr, 1.0)
    synthetic_bad_df = simulate_from_corr(distorted_corr, N_SYNTH, mean_vec, std_vec, seed=3)

    run_scenario("A - Dummy synthetic preserves REAL human structure", human_df, r_human, synthetic_good_df, n_human)
    run_scenario("B - Dummy synthetic distorts Neuroticism relationships", human_df, r_human, synthetic_bad_df, n_human)

    print("\n" + "="*70)
    print("HUMAN DATA IS REAL. SYNTHETIC DATA IS STILL A DUMMY STAND-IN.")
    print("Still need to replace synth_df with real SurveyMind output once available.")
    print("="*70)

(1015341, 110)
   EXT1  EXT2  EXT3  EXT4  EXT5  EXT6  EXT7  EXT8  EXT9  EXT10  ...  \
0   4.0   1.0   5.0   2.0   5.0   1.0   5.0   2.0   4.0    1.0  ...   
1   3.0   5.0   3.0   4.0   3.0   3.0   2.0   5.0   1.0    5.0  ...   
2   2.0   3.0   4.0   4.0   3.0   2.0   1.0   3.0   2.0    5.0  ...   
3   2.0   2.0   2.0   3.0   4.0   2.0   2.0   4.0   1.0    4.0  ...   
4   3.0   3.0   3.0   3.0   5.0   3.0   3.0   5.0   3.0    4.0  ...   

              dateload  screenw  screenh  introelapse  testelapse  endelapse  \
0  2016-03-03 02:01:01    768.0   1024.0          9.0       234.0          6   
1  2016-03-03 02:01:20   1360.0    768.0         12.0       179.0         11   
2  2016-03-03 02:01:56   1366.0    768.0          3.0       186.0          7   
3  2016-03-03 02:02:02   1920.0   1200.0        186.0       219.0          7   
4  2016-03-03 02:02:57   1366.0    768.0          8.0       315.0         17   

   IPC  country  lat_appx_lots_of_err  long_appx_lots_of_err  
0    1       G